# Oryza Vision — Export LiteRT / TFLite (Penyakit + Hama)
**Tanpa training ulang.** Notebook ini memuat checkpoint PyTorch yang sudah ada di Google Drive, mengonversi dua model secara **terpisah**, membandingkan prediksi PyTorch vs LiteRT, lalu menyiapkan paket Flutter.

**Folder sumber (sudah diperiksa):** `MyDrive/Oryza-Vision/models/` dan `MyDrive/Oryza-Vision/pest-training/`.

**Penting:** Konversi `litert-torch` masih dapat gagal pada operator tertentu. Notebook **tidak akan menganggap model berhasil** jika validasi gagal. Mulai dari model float32; optimasi FP16/INT8 dilakukan setelah konversi dan validasi berhasil.

In [ ]:
#@title 1. Instal dependensi converter (jalankan sekali)
%pip -q install "litert-torch" "ai-edge-litert" pandas numpy pillow
print("Dependensi terpasang. Jika Colab meminta restart runtime, restart lalu lanjut dari sel berikutnya.")

In [ ]:
#@title 2. Mount Drive dan tentukan folder
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, sys, time, shutil, traceback, zipfile
import numpy as np
import torch
from torch import nn
from torchvision import models
ROOT = Path('/content/drive/MyDrive/Oryza-Vision')
assert ROOT.exists(), f'Folder tidak ditemukan: {ROOT}. Periksa akun Google yang dipakai saat mount.'
OUT = ROOT / 'tflite-export'
OUT.mkdir(parents=True, exist_ok=True)
DISEASE_CKPT = ROOT/'models'/'mobilenet_v3_large_best.pth'
PEST_CKPT = ROOT/'pest-training'/'pest_efficientnet_b0_best.pth'
for p in (DISEASE_CKPT, PEST_CKPT):
    assert p.is_file(), f'Checkpoint tidak ada: {p}'
print('PyTorch:', torch.__version__)
print('Sumber penyakit:', DISEASE_CKPT)
print('Sumber hama:', PEST_CKPT)
print('Output:',OUT)

In [ ]:
#@title 3. Load model dan metadata kelas (tanpa akses internet untuk pretrained weights)
# Dua notebook training lama punya nama field checkpoint berbeda.
def load_checkpoint(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except Exception as e:
        raise RuntimeError(f'Gagal baca checkpoint {path.name}: {e}')

def build_model(arch, n):
    if arch == 'mobilenet_v3_large':
        m = models.mobilenet_v3_large(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n)
    elif arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n)
    else:
        raise ValueError(f'Architecture tidak didukung: {arch}')
    return m.eval()

def prepare(path, expected_arch):
    checkpoint = load_checkpoint(path)
    classes = checkpoint.get('classes')
    if not isinstance(classes, (list, tuple)) or not classes:
        raise ValueError(f'Checkpoint {path.name} tidak menyimpan classes.')
    state = checkpoint.get('model_state_dict', checkpoint.get('model_state'))
    if not isinstance(state, dict):
        raise ValueError(f'Format state dict tidak ditemukan di {path.name}')
    arch = checkpoint.get('architecture', expected_arch)
    if arch != expected_arch:
        raise ValueError(f'Arsitektur checkpoint {path.name}: {arch}, expected {expected_arch}')
    model = build_model(arch, len(classes))
    model.load_state_dict(state, strict=True)
    size = int(checkpoint.get('img_size', checkpoint.get('image_size', 224)))
    normalization = checkpoint.get('normalization', {})
    mean = normalization.get('mean', [0.485,0.456,0.406])
    std = normalization.get('std', [0.229,0.224,0.225])
    return {'model':model, 'architecture':arch, 'classes':list(classes),
            'size':size, 'mean':mean, 'std':std}

tasks = {
 'disease': prepare(DISEASE_CKPT, 'mobilenet_v3_large'),
 'pest': prepare(PEST_CKPT, 'efficientnet_b0')
}
for name,t in tasks.items():
    print(name, '|',t['architecture'],'|',t['size'],'|',t['classes'])

In [ ]:
#@title 4. Export secara terpisah: PyTorch → LiteRT (.tflite)
import litert_torch
from datetime import datetime, timezone

# Biar reproducible. Input float32 dengan layout NHWC (format yang mudah untuk Flutter).
torch.manual_seed(42)
np.random.seed(42)
results={}
for name, task in tasks.items():
    print(f'\n===== {name.upper()} =====', flush=True)
    tflite_path = OUT / f'{name}_model.tflite'
    # Hapus output lama agar kegagalan baru tidak terbaca sebagai sukses.
    tflite_path.unlink(missing_ok=True)
    model = task['model'].cpu().eval()
    h = w = task['size']
    x = torch.rand(1,h,w,3,dtype=torch.float32)
    # Wrapper menyesuaikan input NHWC -> NCHW tanpa mengubah bobot model.
    wrapped = litert_torch.to_channel_last_io(model, args=[0])
    item={'status':'failed','path':str(tflite_path),'error':None,'validation':None}
    try:
        with torch.no_grad():
            edge = litert_torch.convert(wrapped, (x,))
            edge.export(str(tflite_path))
        if not tflite_path.is_file() or tflite_path.stat().st_size == 0:
            raise RuntimeError('File .tflite tidak berhasil disimpan')
        item['status']='converted_pending_validation'
        item['size_mb']=round(tflite_path.stat().st_size/1024**2,2)
        # Simpan model converter sementara untuk sel validasi.
        item['_edge']=edge
        print('Export OK:',tflite_path.name,item['size_mb'],'MB')
    except Exception as e:
        item['error']=repr(e)
        print('KONVERSI GAGAL:',repr(e))
    results[name]=item

In [ ]:
#@title 5. Validasi keluaran model setelah konversi
# Perbandingan dilakukan untuk 10 input sintetis, bukan pengganti evaluasi akurasi dataset.
# Bila prediksi bergeser/selisih numerik besar, model tidak dinyatakan siap deployment.
for name,task in tasks.items():
    r=results[name]
    if r['status'] != 'converted_pending_validation':
        continue
    try:
        model=task['model'].eval()
        edge=r['_edge']
        differences=[]; equal_top1=0
        for i in range(10):
            x=torch.rand(1,task['size'],task['size'],3,dtype=torch.float32)
            with torch.no_grad():
                pt=model(x.permute(0,3,1,2).contiguous()).detach().cpu().numpy()
                converted=np.asarray(edge(x))
            if pt.shape != converted.shape:
                raise RuntimeError(f'Shape mismatch: PyTorch {pt.shape} vs LiteRT {converted.shape}')
            diff=float(np.max(np.abs(pt-converted)))
            differences.append(diff)
            equal_top1 += int(np.argmax(pt,axis=1)[0] == np.argmax(converted,axis=1)[0])
        maxdiff=max(differences)
        agreement=equal_top1/10
        passed=(maxdiff <= 0.05 and agreement == 1.0)
        r['validation']={'max_abs_logit_difference':maxdiff,
                         'top1_agreement_on_synthetic_inputs':agreement,
                         'samples':10,'passed':bool(passed)}
        r['status']='validated' if passed else 'validation_failed'
        print(f"{name}: {'PASS' if passed else 'FAIL'} | max diff {maxdiff:.6f} | agreement {agreement:.0%}")
    except Exception as e:
        r['status']='validation_failed'
        r['error']=repr(e)
        print(name,'VALIDASI GAGAL:',repr(e))
    finally:
        r.pop('_edge',None)

In [ ]:
#@title 6. Metadata Flutter dan berkas label
# Resize pendek ke 256 lalu center-crop 224 mengikuti kode evaluation training.
# Jika image_size bukan 224, resize 256 perlu disesuaikan secara proporsional.
from datetime import datetime, timezone
summary={}
for name,task in tasks.items():
    r=results[name]
    metadata={
       'task':name,
       'architecture':task['architecture'],
       'input_layout':'NHWC',
       'input_dtype':'float32',
       'input_shape':[1,task['size'],task['size'],3],
       'image_preprocessing':{
           'read':'RGB; respect EXIF orientation',
           'resize_shorter_side':round(task['size']*256/224),
           'center_crop':[task['size'],task['size']],
           'pixel_scale':'divide RGB values by 255.0',
           'channel_order':'RGB',
           'normalize_mean':task['mean'],
           'normalize_std':task['std'],
           'formula':'(pixel/255 - mean[channel]) / std[channel]'
       },
       'output':'raw logits; softmax then argmax',
       'labels':task['classes'],
       'status':r['status'],
       'note':'Model tidak mengandung preprocessing. Flutter WAJIB meniru metadata.'
    }
    (OUT/f'{name}_labels.json').write_text(json.dumps(task['classes'],indent=2),encoding='utf-8')
    (OUT/f'{name}_preprocessing.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    summary[name]={k:v for k,v in r.items() if k!='_edge'}
    print(name,'→',r['status'])
(OUT/'conversion_report.json').write_text(json.dumps(summary,indent=2,default=str),encoding='utf-8')

In [ ]:
#@title 7. Buat paket deployment HANYA kalau dua model lolos validasi
ready=all(results[n]['status']=='validated' for n in tasks)
archive=ROOT/'Oryza_Vision_TFLite_Deployment.zip'
if ready:
    include=[]
    for n in tasks:
        include.extend([OUT/f'{n}_model.tflite',OUT/f'{n}_labels.json',OUT/f'{n}_preprocessing.json'])
    include.append(OUT/'conversion_report.json')
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in include:
            z.write(p,arcname=p.name)
    print('SUKSES: ',archive)
    print('Copy hasil .tflite dan file metadata ke assets/models/ Flutter.')
else:
    archive.unlink(missing_ok=True)
    print('Belum siap deployment. Tidak membuat ZIP palsu.')
    for name,r in results.items():
        print(name,r['status'],r.get('error'),r.get('validation'))
    print('Kirim output error dari sel 4/5 jika converter belum mendukung operator model.')

## Apa yang harus diuji setelah berhasil?
- Kedua model diload satu per satu di Flutter; jangan aktifkan dua interpreter bersamaan bila tidak dibutuhkan.
- Samakan **urutan label**, *resize → center crop*, dan normalisasi ImageNet.
- Jalankan uji foto nyata dan ukur latensi/RAM di perangkat Android target.
- Notebook ini **tidak** menghasilkan FP16/INT8 otomatis. Lakukan quantization belakangan, lalu ukur kembali kecocokan prediksi dan akurasi.